# Custom A/B Experiment Analysis

This notebook allows you to:
1. **Create custom scenarios** - Define your own experiment parameters
2. **Use your own data** - Supply custom assignments and events
3. **Analyze results** - Run statistical tests on your data



---

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd

from simulation import (
    ExperimentConfig,
    ExperimentResults,
    ExperimentReporter,
    Hypothesis,
    ExperimentDesign,
    ExperimentData,
    run_experiment,
    design_experiment,
    measure_metrics,
    analyze_results,
    make_decision,
)
from analysis import (
    calculate_sample_size,
    calculate_mde,
    two_proportion_z_test,
    two_sample_t_test,
)

print("Setup complete!")

---
## Option 1: Create a Custom Scenario

Modify the parameters below to simulate your own experiment.

In [ ]:
# =============================================================================
# CUSTOMIZE YOUR EXPERIMENT HERE
# =============================================================================

config = ExperimentConfig(
    num_users=10000,              # Total users in experiment
    control_conversion_rate=0.10, # Baseline conversion rate (10%)
    treatment_lift=0.15,          # Expected relative lift (15% = 10% -> 11.5%)
    control_avg_order_value=45.0, # Average order value in control
    treatment_aov_lift=0.0,       # Relative lift in AOV (0% = no change)
    daily_traffic=1000,           # Users per day (for duration estimate)
    seed=42,                      # Random seed (change for different results)
)

# Run the experiment
results = run_experiment(config)

# Display results
ExperimentReporter(results).print_full_report()

---
## Option 2: Use Your Own Data

Supply your own assignments and events data to analyze a real experiment.

### Required Data Format

**Assignments DataFrame** (`users_df`):
| Column | Type | Description |
|--------|------|-------------|
| `user_id` | str | Unique user identifier |
| `variant` | str | "control" or "treatment" |

**Events DataFrame** (`events_df`):
| Column | Type | Description |
|--------|------|-------------|
| `user_id` | str | User who triggered event |
| `converted` | int | 1 if conversion, 0 otherwise |
| `order_value` | float | Revenue amount (optional) |

In [ ]:
# =============================================================================
# EXAMPLE: Create sample data (replace with your own)
# =============================================================================

# Option A: Load from CSV files
# users_df = pd.read_csv('your_assignments.csv')
# events_df = pd.read_csv('your_events.csv')

# Option B: Create manually
users_df = pd.DataFrame({
    'user_id': [f'user_{i}' for i in range(1000)],
    'variant': ['control'] * 500 + ['treatment'] * 500
})

# Simulate some conversions (replace with your real data)
np.random.seed(42)
events = []
for _, user in users_df.iterrows():
    rate = 0.10 if user['variant'] == 'control' else 0.12
    if np.random.random() < rate:
        events.append({
            'user_id': user['user_id'],
            'converted': 1,
            'order_value': np.random.lognormal(3.5, 0.5)
        })

events_df = pd.DataFrame(events)

print(f"Users: {len(users_df):,}")
print(f"Events: {len(events_df):,}")
print(f"\nAssignments sample:")
display(users_df.head())
print(f"\nEvents sample:")
display(events_df.head())

In [ ]:
# =============================================================================
# ANALYZE YOUR CUSTOM DATA
# =============================================================================

def analyze_custom_data(
    users_df: pd.DataFrame,
    events_df: pd.DataFrame,
    baseline_rate: float = None,
    expected_lift: float = 0.10,
):
    """
    Analyze your own experiment data.
    
    Args:
        users_df: DataFrame with 'user_id' and 'variant' columns
        events_df: DataFrame with 'user_id', 'converted', and optionally 'order_value'
        baseline_rate: Expected baseline rate (auto-calculated if None)
        expected_lift: Expected relative lift for power analysis
    """
    # Validate data
    assert 'user_id' in users_df.columns, "users_df must have 'user_id' column"
    assert 'variant' in users_df.columns, "users_df must have 'variant' column"
    assert 'user_id' in events_df.columns, "events_df must have 'user_id' column"
    
    # Add converted column if missing
    if 'converted' not in events_df.columns:
        events_df = events_df.copy()
        events_df['converted'] = 1
    
    # Calculate actual baseline if not provided
    control_users = users_df[users_df['variant'] == 'control']['user_id']
    control_conv = events_df[events_df['user_id'].isin(control_users)]['user_id'].nunique()
    actual_baseline = control_conv / len(control_users) if len(control_users) > 0 else 0.1
    baseline_rate = baseline_rate or actual_baseline
    
    # Create hypothesis and design
    hypothesis = Hypothesis(
        name="custom_experiment",
        description="Custom experiment analysis",
        baseline_rate=baseline_rate,
        expected_lift=expected_lift,
    )
    
    config = ExperimentConfig(
        num_users=len(users_df),
        control_conversion_rate=baseline_rate,
        treatment_lift=expected_lift,
    )
    
    design = design_experiment(hypothesis, config)
    
    # Compute metrics and run analysis
    metrics = measure_metrics(users_df, events_df)
    conversion_test, revenue_test = analyze_results(users_df, events_df, metrics)
    recommendation = make_decision(conversion_test)
    
    # Build results
    results = ExperimentResults(
        design=design,
        data=ExperimentData(users=users_df, events=events_df, config=config),
        conversion_test=conversion_test,
        revenue_test=revenue_test,
        recommendation=recommendation,
    )
    
    return results


# Run analysis on your data
results = analyze_custom_data(
    users_df=users_df,
    events_df=events_df,
    expected_lift=0.10,  # What lift were you hoping to detect?
)

ExperimentReporter(results).print_full_report()

---
## Option 3: Quick Statistical Tests

Run individual statistical tests on your numbers directly.

In [ ]:
# =============================================================================
# CONVERSION RATE TEST (Two-Proportion Z-Test)
# =============================================================================

# Enter your numbers here
control_conversions = 500
control_total = 5000
treatment_conversions = 550
treatment_total = 5000

result = two_proportion_z_test(
    control_conversions=control_conversions,
    control_total=control_total,
    treatment_conversions=treatment_conversions,
    treatment_total=treatment_total,
)

print(result)

In [ ]:
# =============================================================================
# CONTINUOUS METRIC TEST (Welch's T-Test)
# =============================================================================

# Enter your arrays here (or load from data)
np.random.seed(42)
control_values = np.random.lognormal(3.5, 0.5, size=500)   # e.g., revenue per user
treatment_values = np.random.lognormal(3.6, 0.5, size=500) # slightly higher

result = two_sample_t_test(
    control_values=control_values,
    treatment_values=treatment_values,
)

print(result)

---
## Option 4: Power Analysis Calculator

Plan your experiment before running it.

In [ ]:
# =============================================================================
# SAMPLE SIZE CALCULATOR
# =============================================================================

baseline_rate = 0.10       # Current conversion rate (10%)
minimum_detectable_effect = 0.10  # Smallest lift worth detecting (10% relative)
daily_traffic = 5000       # Users per day

# Calculate required sample size
required_n = calculate_sample_size(
    baseline_rate=baseline_rate,
    minimum_detectable_effect=minimum_detectable_effect,
)

total_users = required_n * 2
days_needed = total_users / daily_traffic

print(f"Power Analysis Results")
print(f"=" * 40)
print(f"Baseline rate:     {baseline_rate:.1%}")
print(f"Target rate:       {baseline_rate * (1 + minimum_detectable_effect):.1%}")
print(f"MDE:               {minimum_detectable_effect:.1%} relative lift")
print(f"")
print(f"Required sample:   {required_n:,} per variant")
print(f"Total users:       {total_users:,}")
print(f"Estimated runtime: {days_needed:.0f} days")

In [ ]:
# =============================================================================
# MDE CALCULATOR (What can I detect with N users?)
# =============================================================================

available_users_per_variant = 5000
baseline_rate = 0.10

mde = calculate_mde(
    sample_size_per_variant=available_users_per_variant,
    baseline_rate=baseline_rate,
)

print(f"MDE Calculator Results")
print(f"=" * 40)
print(f"Available users:   {available_users_per_variant:,} per variant")
print(f"Baseline rate:     {baseline_rate:.1%}")
print(f"")
print(f"Minimum Detectable Effect: {mde:.1%} relative lift")
print(f"(Smallest effect you can reliably detect at 80% power)")

---
## Option 5: Load Data from Files

Load your experiment data from CSV files.

In [ ]:
# =============================================================================
# LOAD FROM CSV FILES
# =============================================================================

# Uncomment and modify paths to load your data:

# users_df = pd.read_csv('path/to/assignments.csv')
# events_df = pd.read_csv('path/to/events.csv')

# Example CSV format for assignments.csv:
# user_id,variant
# user_001,control
# user_002,treatment
# ...

# Example CSV format for events.csv:
# user_id,converted,order_value
# user_001,1,45.99
# user_005,1,32.50
# ...

print("Modify the paths above to load your CSV files")

In [ ]:
# =============================================================================
# ANALYZE LOADED DATA
# =============================================================================

# After loading your data, run:
# results = analyze_custom_data(users_df, events_df)
# ExperimentReporter(results).print_full_report()